# 🏠 Airbnb Price Predictor

This notebook walks through the full pipeline to predict Airbnb listing prices based on location and property characteristics.

**Goal:** Given features like neighbourhood, room type, and reviews — predict the nightly price.

**Approach:**
1. Load and explore the data
2. Clean and preprocess
3. Train 3 models (Linear Regression, Random Forest, LightGBM)
4. Compare results

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

print('All imports successful!')

## 2. Load Data

The dataset uses `latin-1` encoding (common in European datasets) and has mixed types in some columns, so we use `low_memory=False`.

In [ ]:
DATA_PATH = Path('../data/raw')

df = pd.read_csv(DATA_PATH / 'Listings.csv', encoding='latin-1', low_memory=False)
df.columns = df.columns.str.strip()  # remove hidden whitespace from column names

print(f'Shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
df.info()

In [ ]:
# Check missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
# Price distribution (before cleaning)
print(df['price'].dtype)
print(df['price'].value_counts().head(10))

In [ ]:
# Room type distribution
df['room_type'].value_counts().plot(kind='bar', figsize=(8, 4), title='Room Type Distribution')
plt.tight_layout()
plt.show()

## 4. Data Cleaning

### 4.1 Remove non-numeric prices

Some rows have non-numeric price values (e.g. empty strings, symbols). We identify and remove them.

In [ ]:
mask = pd.to_numeric(df['price'], errors='coerce').isna()
print(f'Non-numeric price rows: {mask.sum()}')
print('Unique non-numeric values:', df[mask]['price'].unique())

df = df[~mask].copy()
df['price'] = df['price'].astype(float)

### 4.2 Remove zero prices

A listing with price = 0 is almost certainly corrupt data, not a free listing.

In [ ]:
print(f'Zero price rows: {(df["price"] == 0).sum()}')
df = df[df['price'] > 0]

### 4.3 Log-transform the price

Prices are heavily right-skewed (a few very expensive listings). Log-transforming makes the distribution more normal, which helps regression models.

In [ ]:
df['price_log'] = np.log1p(df['price'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['price'], bins=100, color='steelblue')
axes[0].set_title('Price (original)')
axes[0].set_xlabel('Price')

axes[1].hist(df['price_log'], bins=100, color='coral')
axes[1].set_title('Price (log-transformed)')
axes[1].set_xlabel('log(Price)')

plt.tight_layout()
plt.show()

## 5. Feature Engineering

### 5.1 Select features

In [ ]:
FEATURES = [
    'latitude', 'longitude',                   # geographic coordinates
    'neighbourhood', 'district', 'city',       # location categories
    'room_type', 'property_type',              # type of space
    'accommodates', 'bedrooms',                # size
    'minimum_nights',                          # booking restriction
    'host_is_superhost',                       # host quality
    'review_scores_rating', 'review_scores_location',  # reviews
    'instant_bookable'                         # convenience
]
TARGET = 'price_log'

### 5.2 Encode boolean columns

`host_is_superhost` and `instant_bookable` are stored as `t`/`f` strings. We convert them to `1`/`0`.

In [ ]:
bool_cols = ['host_is_superhost', 'instant_bookable']
df[bool_cols] = df[bool_cols].apply(lambda col: col.map({'t': 1, 'f': 0}))

### 5.3 Drop rows with missing values in selected features

In [ ]:
df_model = df[FEATURES + [TARGET]].dropna()
print(f'Rows after dropna: {df_model.shape[0]}')

X = df_model[FEATURES]
y = df_model[TARGET]

### 5.4 Encode categorical columns

We use `OrdinalEncoder` for high-cardinality columns like `neighbourhood` and `district`.

> **Why not OneHotEncoder?** Neighbourhood alone can have 50-100+ unique values, creating hundreds of sparse columns. OrdinalEncoder keeps one column per feature and works well with tree-based models.

In [ ]:
cat_cols = ['neighbourhood', 'district', 'city', 'room_type', 'property_type']

ct = ColumnTransformer(transformers=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
], remainder='passthrough')

X_encoded = ct.fit_transform(X)
print(f'Encoded shape: {X_encoded.shape}')

## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size:  {X_test.shape[0]}')

## 7. Model Training & Evaluation

We train 3 models and compare them using **MAE**, **RMSE**, **R²**, and **10-fold cross-validation**.

Since we trained on `price_log`, all metrics are in log scale. To interpret in real price:
```python
real_price = np.expm1(predicted_log_price)
```

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        cv_scores = cross_val_score(model, X_train, y_train, cv=10, scoring='r2')

    print(f'\n--- {name} ---')
    print(f'MAE:            {mae:.4f}')
    print(f'RMSE:           {rmse:.4f}')
    print(f'R²:             {r2:.4f}')
    print(f'CV R² (10-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

    return {'name': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv_scores.mean()}

### 7.1 Linear Regression

Our baseline model. Simple, fast, and interpretable.

In [ ]:
lr_results = evaluate_model(
    'Linear Regression',
    LinearRegression(),
    X_train, X_test, y_train, y_test
)

### 7.2 Random Forest

An ensemble of decision trees. Handles non-linear relationships and categorical features well.

In [ ]:
rf_results = evaluate_model(
    'Random Forest',
    RandomForestRegressor(n_estimators=100, random_state=42),
    X_train, X_test, y_train, y_test
)

### 7.3 LightGBM

Gradient boosting model. Fast, memory-efficient, and typically achieves the best results.

In [ ]:
lgbm_results = evaluate_model(
    'LightGBM',
    LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
    X_train, X_test, y_train, y_test
)

## 8. Model Comparison

In [ ]:
results = pd.DataFrame([lr_results, rf_results, lgbm_results])
results.set_index('name', inplace=True)
print(results.round(4))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ['MAE', 'RMSE', 'R2']
colors  = ['steelblue', 'coral', 'seagreen']

for ax, metric, color in zip(axes, metrics, colors):
    results[metric].plot(kind='bar', ax=ax, color=color, title=metric)
    ax.set_xticklabels(results.index, rotation=20, ha='right')
    ax.set_ylabel(metric)

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Conclusions

- **Linear Regression** is the weakest model but gives a useful baseline and is fully interpretable.
- **Random Forest** improves significantly by capturing non-linear relationships between features and price.
- **LightGBM** achieves the best performance — fastest to train and most accurate.

### Next steps
- Hyperparameter tuning (GridSearchCV or Optuna) on LightGBM
- Target encoding for `neighbourhood` and `district` instead of OrdinalEncoder
- Add more features: `amenities` count, host listing count, proximity to landmarks
- Deploy the best model as an API with FastAPI